# Cross-Lingual GraphRAG Pipeline on Kaggle
Ensure you have set the Accelerator to **GPU T4 x2** before running this notebook.
This notebook runs the complete pipeline with **Neo4j Graph Integration** and **Persian Translation**.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import sys
# Ensure we are in the root working directory and remove any existing repo
os.chdir('/kaggle/working')
!rm -rf /kaggle/working/MedRAG

# Clone the MedRAG repository from GitHub
!git clone https://github.com/TeleEng/MedRAG.git

# Change working directory into the repo
os.chdir('MedRAG')

print("=== Repository Version Info ===")
!git log -1 --format="Commit: %h | Date: %cd"
print("===============================")

# Install dependencies
!pip install -q -r requirements.txt

# Add repo root to Python path so 'from src...' imports work
if '.' not in sys.path:
    sys.path.insert(0, '.')


Cloning into 'MedRAG'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 164 (delta 95), reused 113 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 135.50 KiB | 13.55 MiB/s, done.
Resolving deltas: 100% (95/95), done.
=== Repository Version Info ===
Commit: f833df0 | Date: Sat Aug 29 06:08:47 2026 -0500
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 116.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

### Step 0: Neo4j Secrets Configuration
Load Neo4j credentials from Kaggle Secrets so they are available to the pipeline.

In [2]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
try:
    os.environ["NEO4J_URI"] = user_secrets.get_secret("NEO4J_URI")
    os.environ["NEO4J_USERNAME"] = user_secrets.get_secret("NEO4J_USERNAME")
    os.environ["NEO4J_PASSWORD"] = user_secrets.get_secret("NEO4J_PASSWORD")
    print("Neo4j Secrets Loaded Successfully!")
except Exception as e:
    print("Warning: Please configure NEO4J_URI, NEO4J_USERNAME, and NEO4J_PASSWORD in Kaggle Secrets.")

Neo4j Secrets Loaded Successfully!


### Step 1: Data Preparation
Download the `medalpaca` dataset and process it into chunks.

In [3]:
from src.data_prep import load_and_prepare_data
load_and_prepare_data()

Loading dataset medalpaca/medical_meadow_wikidoc...


README.md: 0.00B [00:00, ?B/s]

medical_meadow_wikidoc.json:   0%|          | 0.00/10.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Processing documents and creating training pairs...
Saved 9373 documents to /kaggle/working/MedRAG/data/processed/documents.json
Saved 9373 training samples to /kaggle/working/MedRAG/data/processed/train_data.json


### Step 2: Indexing (FAISS, BM25, and Neo4j Graph)
Build the Dense/Sparse indexes and push entities to Neo4j AuraDB.

In [4]:
from src.indexer import build_indexes
from src.graph_indexer import GraphIndexer

# 1. Local Indexes
build_indexes()

# 2. Graph Database Indexing
g_indexer = GraphIndexer()
g_indexer.build_graph()
g_indexer.close()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Building Dense Index for 9373 documents...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/293 [00:00<?, ?it/s]

Dense Index saved to /kaggle/working/MedRAG/data/processed/faiss_index.bin
Building BM25 Sparse Index...
BM25 Index saved to /kaggle/working/MedRAG/data/processed/bm25_index.pkl
Initializing GraphIndexer in 'auto' mode...
Attempting to connect to Neo4j at neo4j+s://2fa211c8.databases.neo4j.io...
Neo4j connection failed: Failed to DNS resolve address 2fa211c8.databases.neo4j.io:7687: [Errno -2] Name or service not known. Falling back to KùzuDB.
Connecting to local KùzuDB at /kaggle/working/MedRAG/data/processed/kuzu_db...
Creating KùzuDB Schema...
Extracting entities using SpaCy and building Graph (kuzu)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 57.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel

### Step 3: QLoRA Fine-Tuning
Instruction-tune the base model.

In [5]:
from src.trainer import train_model
train_model()

Loading tokenizer and configuring formatting...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/9373 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting QLoRA Fine-Tuning...


Step,Training Loss
10,2.231372
20,2.129537
30,1.967055
40,2.113220
50,2.024053


Saving trained adapter...
Training complete! Run generator.py for end-to-end RAG.


### Step 4: End-to-End Cross-Lingual Generation
Run the full pipeline. The system will translate Persian -> English, query Neo4j+FAISS, generate in English, and translate back to Persian.

In [6]:
from src.generator import MedRAGPipeline

pipeline = MedRAGPipeline()

query_pes = "علائم بیماری دیابت چیست؟"
res_pes, res_eng, docs = pipeline.answer_query(query_pes)

print("\n==========================")
print("MEDRAG PERSIAN ANSWER:")
print("==========================")
print(res_pes)

print("\n==========================")
print("ORIGINAL ENGLISH ANSWER:")
print("==========================")
print(res_eng)

Loading documents and indexes...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

...

Loading trained QLoRA adapter...


Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'repetition_penalty', 'top_p', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



[0] Translating User Query to English...
Executing Graph Retrieval...


/usr/local/lib/python3.12/dist-packages/spacy/util.py:1800: UserWarning: [W111] Jupyter notebook detected: if using `prefer_gpu()` or `require_gpu()`, include it in the same cell right before `spacy.load()` to ensure that the model is loaded on the correct device. More information: http://spacy.io/usage/v3#jupyter-notebook-gpu
  warnings.warn(Warnings.W111)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Executing FAISS and BM25 Hybrid Search...

[3] Translating English Response back to Persian...

MEDRAG PERSIAN ANSWER:
پیش بینی بالینی دیابت شامل سه دسته اصلی است: دیابت انسپیدوس، دیابت نوع یک، و دیابت نوع دو. دیابت انسپیدوس دارای ادرار بیش از حد و تشنه شدید است. بیماران جوان تر اغلب از طریق ادرار مکرر دچار تهوع یا استفراغ می شوند. بزرگسالان مسن مبتلا به دیابت انسپیدوس معمولاً هیچ نشانه دیگری را به جز افزایش ادرار و تشنه نشان نمی دهند. دیابت نوع یک دارای علائم کلاسیک مانند کاهش وزن غیر قابل توضیح، خستگی، ضعف بینایی، خشک دهان، مکرر ادرار و تشنه شدید است. بیماران نیز تمایل دارند که بیشتر از همیشه تحریک شوند. دیابت نوع دو معمولاً باعث بروز علائم کمتر قابل توجهی مانند احساس خستگی، داشتن مشکل کنترل قند خون، تشنگی مکرر، نیاز به رفتن به حمام در طول شب و تجربه کردن ضعف بینایی است.

ORIGINAL ENGLISH ANSWER:
The clinical presentation of diabetes includes three main categories: Diabetes Insipidus, Type I Diabetes Mellitus, and Type II Diabetes Mellitus.
Diabetes Insipidus presents with excessive 